# 03 · Streaming the Graph, the Gateway, and the Fixed Chain

Notebook `01` ran the graph synchronously. The real app streams it to a React UI
over **Server-Sent Events (SSE)** so the engineer watches routing and tokens
appear live. This notebook covers three things:

1. **The SSE event stream** — the sequence of events the orchestrator emits.
2. **AgentCore Gateway** — how `kb_search` and `web_search_eol` route tool calls
   through one governed MCP endpoint (with a direct fallback).
3. **The Integration Impact Assessment** — a *fixed* specialist chain that reuses
   the same specialists but is **not** part of the planned chat graph.

## 0 · Setup

In [1]:
import os, sys, json, threading
from queue import Queue, Empty
from pathlib import Path

# Standalone: use the vendored mei_core package + bundled data in this folder.
# No dependency on the parent app's agent/ or api/ — this folder is portable.
NB_DIR = Path.cwd() if (Path.cwd() / 'mei_core').exists() else Path.cwd() / 'notebooks'
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))
REPO_ROOT = NB_DIR  # alias: bundled demo-crops/ and evals/ live under NB_DIR
_bundled = NB_DIR / 'data' / 'synthetic-parts-data'
if _bundled.exists():
    os.environ['PARTS_DATA_DIR'] = str(_bundled)

os.environ.setdefault('DATASET', 'synthetic')
os.environ.setdefault('BEDROCK_REGION', 'us-east-1')

from mei_core import dataset_profiles
dataset_profiles.resolve()
print('gateway configured:', bool(os.environ.get('AGENTCORE_GATEWAY_ID')))

gateway configured: False


## 1 · The SSE event stream

`api/services/stream_adapter.py` runs the *same* planner → specialists →
supervisor orchestration as `run_multi_agent`, but instead of returning one dict
it pushes **events** onto a queue as it goes. `run_agent_with_events` is the
function; it emits, in order:

| event | when |
|---|---|
| `routing` | planner finished — carries the plan + ordered specialists |
| `specialist_start` | a specialist begins |
| `content_delta` | one streamed token chunk from the current specialist |
| `content` | the specialist's full text (for the collapsed history view) |
| `specialist_done` | a specialist finished |
| `synthesis` | the final merged answer (only when 2+ specialists ran) |
| `images` | part datasheet/photo refs to render inline |
| `done` | stream complete |

We run it on a background thread (exactly like `agent_runner.py` does) and drain
the queue, printing a compact trace. Token deltas are collapsed so the log stays
readable.

In [2]:
from mei_core.stream_adapter import run_agent_with_events

def stream_query(user_input: str, image_b64=None):
    q = Queue()
    t = threading.Thread(target=run_agent_with_events, args=(user_input, image_b64, q), daemon=True)
    t.start()

    final_text = ''
    delta_counts = {}
    while True:
        try:
            evt = q.get(timeout=180)
        except Empty:
            print('(timed out waiting for events)')
            break
        name, data = evt['event'], evt['data']
        if name == 'routing':
            print(f"[routing]          specialists = {data['specialists']}")
        elif name == 'specialist_start':
            print(f"[specialist_start] {data['name']}")
        elif name == 'content_delta':
            delta_counts[data['specialist']] = delta_counts.get(data['specialist'], 0) + 1
        elif name == 'specialist_done':
            n = delta_counts.get(data['name'], 0)
            print(f"[specialist_done]  {data['name']}  ({n} token chunks streamed)")
        elif name == 'synthesis':
            final_text = data['text']
            print('[synthesis]        (supervisor merged the specialists)')
        elif name == 'content':
            if not final_text:
                final_text = data['text']  # single-specialist case: no synthesis event
        elif name == 'images':
            refs = [f"{i['part_id']}/{i['filename']}" for i in data['images']]
            print(f"[images]           {refs}")
        elif name == 'error':
            print(f"[error]            {data}")
        elif name == 'done':
            print('[done]')
            break
    t.join(timeout=5)
    return final_text

answer = stream_query('Which reaction wheels have at least 0.05 Nms of momentum storage?')
print('\n================ FINAL ANSWER ================\n')
print(answer)

1. CATALOG — discover reaction wheels with momentum storage of at least 0.05 Nms

[routing]          specialists = ['catalog']
[specialist_start] catalog


[specialist_done]  catalog  (148 token chunks streamed)
[synthesis]        (supervisor merged the specialists)
[done]

================ FINAL ANSWER ================

Four of the 7 reaction wheels in the catalog have at least 0.05 Nms of momentum storage. All four are from AnyCompany, and all are Active and in full production. I got this from an exhaustive scan of the catalog on `momentum_storage_nms`.

| Part ID | Name (model) | Momentum (Nms) | Max torque (mNm) | Mass (g) | Supply voltage (V) | Interface | Power (W) | Lead time (weeks) |
|---|---|---|---|---|---|---|---|---|
| `rw_rl_0060` | AnyCompany RW3 0.06 (RW3-0.06) | 0.06 | 20 | 235 | 28 nominal (7.5–36 range) | RS-422 or Redundant RS-485 | 1.3 | 10 |
| `rw_rl_0400` | AnyCompany RW4 0.4 (RW4-0.4) | 0.4 | 100 | 755 | 28 nominal (22–34 range) | RS-422 or Redundant RS-485 | 2.8 | 12 |
| `rw_rl_1000` | AnyCompany RW4 1.0 (RW4-1.0) | 1.0 | 100 | 1400 | 28 nominal (22–34 range) | RS-422 or Redundant RS-485 | 5 | 12 |
| `rw_rl_5000` 

That's the exact event sequence the frontend consumes. `agent_runner.py` wraps
this same queue in an async generator and formats each event as
`event: <type>\ndata: <json>\n\n` SSE text; the `POST /api/chat/stream` route
returns it as a `StreamingResponse`.

Note the **single-specialist** case emits no `synthesis` event — the UI shows the
specialist's streamed text directly, matching the "skip redundant Opus call"
behavior from notebook `01`.

## 2 · AgentCore Gateway routing

Two tools can route through an **AgentCore Gateway** — a single governed MCP
endpoint — when `AGENTCORE_GATEWAY_ID` is set:

- **`kb_search`** → the Gateway's *Managed Knowledge Base* connector (text
  retrieval). Set up by `agentcore/setup_kb_target.py`.
- **`web_search_eol`** → the Gateway's *Web Search* connector (live EOL/market
  data, zero data egress). Set up by `agentcore/setup_web_search.py`.

The pattern inside both tools is the same: build the MCP JSON-RPC payload,
**SigV4-sign** it, `POST` to the Gateway `/mcp` endpoint, and parse the result.
Tool names are discovered once via `tools/list` and cached. If the Gateway isn't
configured (or the call fails), the tool **falls back** — `kb_search` to a direct
Bedrock `Retrieve`, `web_search_eol` to simulated demo results — so nothing breaks.

In [3]:
from mei_core.tools import web_search_eol

# With no AGENTCORE_GATEWAY_ID set this returns simulated demo data; with a
# Gateway configured the same call routes through AgentCore Web Search (live).
res = web_search_eol('AnyCompany RW3 0.06 reaction wheel lead time end of life 2026')
print('result count:', res.get('count'))
print('source      :', res.get('source', res['results'][0]['source'] if res.get('results') else 'n/a'))
for r in res.get('results', [])[:2]:
    print(f"  - {r.get('title')}")

result count: 3
source      : web_search (simulated for demo)
  - AnyCompany Space Systems — Reaction Wheel Product Line
  - DMSMS Alert: Component Lifecycle Monitoring


In [4]:
import os
REGION = os.environ.get('BEDROCK_REGION', 'us-east-1')
# The actual AgentCore Gateway call, reproduced inline (this is the body of
# web_search_eol's Gateway branch in agent/tools.py). The pattern is: build an
# MCP JSON-RPC request, SIGN it with SigV4 for the bedrock-agentcore service,
# POST to the Gateway's /mcp endpoint, and parse the tool result. Tool discovery
# (tools/list) is cached after the first call.
import json, boto3, requests
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

def gateway_web_search(query, gateway_id, region=REGION):
    """Call AgentCore Web Search through the Gateway MCP endpoint (SigV4-signed).
    Returns parsed results, or raises — the real tool wraps this in try/except and
    falls back to simulated results when AGENTCORE_GATEWAY_ID is unset."""
    session = boto3.Session()
    creds = session.get_credentials().get_frozen_credentials()
    url = f'https://{gateway_id}.gateway.bedrock-agentcore.{region}.amazonaws.com/mcp'

    def _signed_post(payload):
        req = AWSRequest(method='POST', url=url, data=json.dumps(payload),
                         headers={'Content-Type': 'application/json'})
        SigV4Auth(creds, 'bedrock-agentcore', region).add_auth(req)
        return requests.post(url, headers=dict(req.headers), data=req.body, timeout=30)

    # 1) tools/list to discover the WebSearch tool name (target-prefixed)
    listed = _signed_post({'jsonrpc': '2.0', 'id': 'list', 'method': 'tools/list', 'params': {}})
    tool_name = 'WebSearch'
    if listed.status_code == 200:
        for t in listed.json().get('result', {}).get('tools', []):
            if 'WebSearch' in t.get('name', ''):
                tool_name = t['name']
                break

    # 2) tools/call the WebSearch tool
    called = _signed_post({'jsonrpc': '2.0', 'id': 'call', 'method': 'tools/call',
                           'params': {'name': tool_name, 'arguments': {'query': query}}})
    called.raise_for_status()
    return called.json().get('result', {})

# Run it live only if a Gateway is configured; otherwise show the documented path.
gateway_id = os.environ.get('AGENTCORE_GATEWAY_ID', '')
if gateway_id:
    result = gateway_web_search('AnyCompany RW3 0.06 reaction wheel lead time EOL 2026', gateway_id)
    content = result.get('content', [])
    print('Gateway WebSearch returned', len(content), 'content item(s). First item:')
    print(json.dumps(content[0] if content else {}, indent=2)[:600])
else:
    print('AGENTCORE_GATEWAY_ID not set — not calling the Gateway live.')
    print('With a Gateway set, gateway_web_search() above runs the real SigV4 MCP call.')
    print('Set it up with: agentcore/setup_web_search.py (+ setup_kb_target.py for KB).')

AGENTCORE_GATEWAY_ID not set — not calling the Gateway live.
With a Gateway set, gateway_web_search() above runs the real SigV4 MCP call.
Set it up with: agentcore/setup_web_search.py (+ setup_kb_target.py for KB).


## 3 · The Integration Impact Assessment: a *fixed* chain, not the graph

The chat graph (notebooks `01`–`02`) is **dynamically planned** — the planner
decides the route per query. The Integration Impact Assessment (IIA) is
different. It reuses the **same specialist factories**, but runs them in a
**fixed, pre-determined order** with **no planner and no Gateway routing
decision**:

```
Knowledge Specialist  ->  Evaluation Specialist  ->  Supervisor
(ICD/datasheet detail)    (structured delta + qual)   (6-section document)
```

Why fixed? Because we already know exactly what a formal IIA document requires —
there's nothing to classify. It's the right tool when the *task shape* is known
in advance, whereas the planner earns its keep when the *query shape* varies.

It lives in `api/services/impact_assessment.py`, completely separate from the
planned graph. The call below runs the real chain (several Bedrock calls; ~1 min).

In [5]:
from mei_core.impact_assessment import run_integration_impact_assessment

iia = run_integration_impact_assessment(reference_id='rw_rl_0060', candidate_id='rw_rl_0030')
print('document_id:', iia['document_id'])
print(f"delta: {iia['matches_count']}/{iia['total_fields']} fields match, {iia['differences_count']} differ\n")
print(iia['markdown'][:1500], '...')

*Routing: No new

 specialist call is needed. This document synthesizes the Knowledge Specialist findings (ICD interface detail) and the Evaluation Specialist findings (field

-by-field comparison and qualification check) provided above.*

# Integration Impact Assessment

**Reference:** AnyCompany RW3 0.06 (rw

_rl_0060). Vendor: AnyCompany. Source: AC-ICD-00068, RW3-0.06 ICD v2.2.
**Candidate:** AnyCompany R

W 0.03 (rw_rl_0030). Vendor: AnyCompany. Source: AC-ICD-00108, RW-0.03 ICD v2.2.

## 1. Interface

...

 indicated. Lifecycle values are identical.
- The only lifecycle difference is the reliability basis. The reference has 930+ cumulative years over

 317 units and is described as the most-flown

 model. The candidate has 

510+ cumulative years over 90 units and shares a common design with RW-0.01.

document_id: IIA-rw_rl_0060-rw_rl_0030-20261008191157
delta: 5/20 fields match, 15 differ

*Routing: No new specialist call is needed. This document synthesizes the Knowledge Specialist findings (ICD interface detail) and the Evaluation Specialist findings (field-by-field comparison and qualification check) provided above.*

# Integration Impact Assessment

**Reference:** AnyCompany RW3 0.06 (rw_rl_0060). Vendor: AnyCompany. Source: AC-ICD-00068, RW3-0.06 ICD v2.2.
**Candidate:** AnyCompany RW 0.03 (rw_rl_0030). Vendor: AnyCompany. Source: AC-ICD-00108, RW-0.03 ICD v2.2.

## 1. Interface Changes

**Pinout: Changes.** The connector changes from 15-pin to 6-pin, so the existing harness is not pin-compatible.

| Item | Reference (rw_rl_0060) | Candidate (rw_rl_0030) |
|---|---|---|
| Connector | 15-pin Micro-D, MIL-DTL-83513 compatible (§6.2, Fig. 5, Table 3) | J1, 6-pin Micro-D, 2 rows × 3 (§6.2, Fig. 2) |
| Ground | Pin 3: Chassis GND (Table 3 excerpt) | Pin 1: GND |
| Power | Not confi

## 4 · Deploying the agent to AgentCore Runtime

Everything above ran the graph **in-process**. In production the same multi-agent
code is hosted on **Amazon Bedrock AgentCore Runtime** — serverless, session-
isolated, and emitting OTel traces that power the managed evaluations in notebook
`05`. This section is the deployment recipe: the container, the build/push, the
Runtime update, the IAM the execution role needs, and the Cedar policy.

These cells are **reference / copy-paste** (shell + file contents shown via
`%%bash --no-exec` style printing), not executed here — deploying rebuilds a
container and mutates a live Runtime, which shouldn't happen as a side effect of
running a notebook. Copy them to a terminal when you're ready.

### 4.1 The container

AgentCore Runtime serves HTTP on `0.0.0.0:8080`. The image is built **for ARM64**
(a Runtime requirement) and bundles both catalogs so either `DATASET` works
self-contained. This is `agentcore/Dockerfile`:

In [6]:
DOCKERFILE = r'''
FROM python:3.12-slim
WORKDIR /app

# Dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Application code
COPY agent/ ./agent/
COPY api/ ./api/
COPY dataset_profiles.py ./dataset_profiles.py

# Synthetic parts catalog + the requirements schema
COPY data/synthetic-parts-data/ ./data/synthetic-parts-data/
COPY data/requirements_schema.json ./data/requirements_schema.json

# AgentCore entry point (HTTP on 8080: /invocations + /ping)
COPY agentcore/serve.py ./serve.py
EXPOSE 8080

# OTel tracing for AgentCore Evaluations
ENV AGENT_OBSERVABILITY_ENABLED=true
ENV OTEL_PYTHON_DISTRO=aws_distro
ENV OTEL_PYTHON_CONFIGURATOR=aws_configurator
ENV OTEL_EXPORTER_OTLP_PROTOCOL=http/protobuf

# Run via opentelemetry-instrument so traces export to CloudWatch
CMD ["opentelemetry-instrument", "python", "serve.py"]
'''
print(DOCKERFILE)


FROM python:3.12-slim
WORKDIR /app

# Dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Application code
COPY agent/ ./agent/
COPY api/ ./api/
COPY dataset_profiles.py ./dataset_profiles.py

# Synthetic parts catalog + the requirements schema
COPY data/synthetic-parts-data/ ./data/synthetic-parts-data/
COPY data/requirements_schema.json ./data/requirements_schema.json

# AgentCore entry point (HTTP on 8080: /invocations + /ping)
COPY agentcore/serve.py ./serve.py
EXPOSE 8080

# OTel tracing for AgentCore Evaluations
ENV AGENT_OBSERVABILITY_ENABLED=true
ENV OTEL_PYTHON_DISTRO=aws_distro
ENV OTEL_PYTHON_CONFIGURATOR=aws_configurator
ENV OTEL_EXPORTER_OTLP_PROTOCOL=http/protobuf

# Run via opentelemetry-instrument so traces export to CloudWatch
CMD ["opentelemetry-instrument", "python", "serve.py"]



### 4.2 Build (ARM64), push to ECR, update the Runtime

Build an **immutable** tag — don't rely on a moving `:latest`, since AgentCore can
retain a previously-resolved digest. The `--platform linux/arm64` flag is
**required**.

In [7]:
DEPLOY_SH = r'''
export TAG=mei-$(date +%Y%m%d-%H%M%S)
export ACCOUNT=<your-account-id>
export ECR=${ACCOUNT}.dkr.ecr.us-east-1.amazonaws.com/mfg-engineering-intelligence

# 1) Build the container for ARM64 (required by AgentCore Runtime)
docker buildx build --platform linux/arm64 -f agentcore/Dockerfile \
  -t mfg-engineering-intelligence:${TAG} --load .

# 2) Push to ECR
aws ecr get-login-password --region us-east-1 \
  | docker login --username AWS --password-stdin ${ACCOUNT}.dkr.ecr.us-east-1.amazonaws.com
docker tag mfg-engineering-intelligence:${TAG} ${ECR}:${TAG}
docker push ${ECR}:${TAG}

# 3) Point the Runtime at the new immutable tag (create or update the
#    AgentCore Runtime via the console or the starter toolkit / control API).
#    See agentcore/deploy_runtime.py for the create/update helper.
'''
print(DEPLOY_SH)


export TAG=mei-$(date +%Y%m%d-%H%M%S)
export ACCOUNT=<your-account-id>
export ECR=${ACCOUNT}.dkr.ecr.us-east-1.amazonaws.com/mfg-engineering-intelligence

# 1) Build the container for ARM64 (required by AgentCore Runtime)
docker buildx build --platform linux/arm64 -f agentcore/Dockerfile \
  -t mfg-engineering-intelligence:${TAG} --load .

# 2) Push to ECR
aws ecr get-login-password --region us-east-1 \
  | docker login --username AWS --password-stdin ${ACCOUNT}.dkr.ecr.us-east-1.amazonaws.com
docker tag mfg-engineering-intelligence:${TAG} ${ECR}:${TAG}
docker push ${ECR}:${TAG}

# 3) Point the Runtime at the new immutable tag (create or update the
#    AgentCore Runtime via the console or the starter toolkit / control API).
#    See agentcore/deploy_runtime.py for the create/update helper.



### 4.3 Runtime execution-role IAM

The Runtime's execution role needs these — note **`s3vectors:QueryVectors`**: a
developer's own broad creds mask its absence locally, but on the Runtime a missing
`QueryVectors` silently degrades image-as-query (§4 of notebook `04`) to text
search and the agent can name the wrong part.

In [8]:
RUNTIME_IAM = {
    'Version': '2012-10-17',
    'Statement': [{
        'Effect': 'Allow',
        'Action': [
            'bedrock:InvokeModel',                 # Claude + Nova MME inference
            'bedrock:Retrieve',                    # managed KB text retrieval
            'bedrock-agentcore:InvokeGateway',     # Gateway (web search + KB)
            's3vectors:QueryVectors',              # image-as-query (REQUIRED)
        ],
        'Resource': '*',
    }],
}
print(json.dumps(RUNTIME_IAM, indent=2))

{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "bedrock:InvokeModel",
        "bedrock:Retrieve",
        "bedrock-agentcore:InvokeGateway",
        "s3vectors:QueryVectors"
      ],
      "Resource": "*"
    }
  ]
}


### 4.4 Cedar policy (human-in-the-loop guardrails)

`agentcore/policy.cedar` expresses the human-in-the-loop rules as Cedar: a memo
can't be generated without a prior evaluation, and every substitution decision
must be logged. (In this asset the rules are enforced at the app/prompt level
today; the Cedar file documents them for attachment to the Runtime's policy
engine.)

In [9]:
POLICY_CEDAR = r'''
// RULE 1: A substitution justification memo must not be generated without a
// prior spec-fit evaluation (score_substitution_fit / compare_parts).
forbid (
    principal,
    action == Action::"generate_memo_without_evaluation",
    resource
);

// RULE 2: Every substitution decision (evaluation, memo, or IIA) must be
// recorded via log_decision — AS9100 8.4 / 8.5.6 traceability.
forbid (
    principal,
    action == Action::"skip_audit_log",
    resource
);

// PERMIT everything else (search, compare, draft, log).
permit ( principal, action, resource );
'''
print(POLICY_CEDAR)


// RULE 1: A substitution justification memo must not be generated without a
// prior spec-fit evaluation (score_substitution_fit / compare_parts).
forbid (
    principal,
    action == Action::"generate_memo_without_evaluation",
    resource
);

// RULE 2: Every substitution decision (evaluation, memo, or IIA) must be
// recorded via log_decision — AS9100 8.4 / 8.5.6 traceability.
forbid (
    principal,
    action == Action::"skip_audit_log",
    resource
);

// PERMIT everything else (search, compare, draft, log).
permit ( principal, action, resource );



## Recap

- The graph streams to the UI as an ordered **SSE event sequence**
  (`routing` → `specialist_*` → `content_delta` → `synthesis` → `images` →
  `done`); `stream_adapter.py` emits it, `agent_runner.py` serves it.
- **AgentCore Gateway** gives `kb_search` and `web_search_eol` one governed MCP
  endpoint, with graceful fallback when it isn't configured.
- The **Integration Impact Assessment** reuses the specialists in a *fixed* chain —
  it is deliberately **not** part of the dynamically-planned chat graph. Planned
  routing for varying queries; fixed chain for a known document task.

That's the full chat multi-agent graph: planned routing, narrow tool-focused
specialists over a deterministic evidence layer, reserved-cost synthesis, and a
live streaming interface — with the engineer as the decision-maker throughout.